In [ ]:
import itertools
from IPython.core.display import Markdown

from library.circuitry import Circuitry
from library.common import Pauli
from library.surface_code.teleportation import TeleportationSurgery
from library.qubit_array import QubitArray

In [ ]:
qubits = QubitArray(dimensions = (9, 5))
circuitry = Circuitry(qubits=qubits)
basis = Pauli.Z

surgery = TeleportationSurgery(qubits, distance = 3, anchor = (1, 1))
surgery.append_movement(circuitry, prepare = basis, measure = basis)

print(qubits.measurements_index)

for qubit, records in qubits.measurements_qubit.items():
    if len(records) > 1:
        patches = { key for key, _ in itertools.groupby(records, key=lambda mr: mr[0]) }

        print(f"Handling {patches} : {records}")
        if len(patches) == 1:
            # Purely in SOURCE or TARGET patches
            stabilizer = records[0].split(":")[-1]
            if stabilizer.startswith(basis.name):
                circuitry.annotate_detector(records[0])

            for rounds in itertools.pairwise(records):
                circuitry.annotate_detector(*rounds)
        elif len(patches) == 2: # Crossing SOURCE&MERGER or MERGER&TARGET patches
            stabilizer = records[0].split(":")[-1]
            if stabilizer.startswith(basis.name):
                circuitry.annotate_detector(records[0])

            for sub_records in [ records[:3], records[3:] ]:
                for rounds in itertools.pairwise(sub_records):
                    circuitry.annotate_detector(*rounds)

circuitry.annotate_detector('S:M1:R2:Z0', 'S:M1:R2:D0', 'S:M1:R2:D1', 'S:M1:R2:D3', 'S:M1:R2:D4')
circuitry.annotate_detector('S:M1:R2:Z2', 'S:M1:R2:D1', 'S:M1:R2:D2')
circuitry.annotate_detector('S:M1:R2:Z3', 'S:M1:R2:D4', 'S:M1:R2:D5', 'S:M1:R2:D7', 'S:M1:R2:D8')

circuitry.annotate_detector('S:M0:R2:Z1', 'M:M1:R0:Z0')

circuitry.annotate_detector('M:M1:R2:Z0', 'S:M1:R2:D6', 'S:M1:R2:D7', 'M:M1:R2:D3', 'M:M1:R2:D4')
circuitry.annotate_detector('M:M1:R2:Z3', 'M:M1:R2:D4', 'M:M1:R2:D5', 'T:M2:R0:Z2')

circuitry.annotate_detector('T:M2:R2:Z0', 'T:M2:R2:D0', 'T:M2:R2:D1', 'T:M2:R2:D3', 'T:M2:R2:D4')
circuitry.annotate_detector('T:M2:R2:Z1', 'T:M2:R2:D6', 'T:M2:R2:D7')
circuitry.annotate_detector('T:M2:R2:Z2', 'T:M2:R2:D1', 'T:M2:R2:D2')
circuitry.annotate_detector('T:M2:R2:Z3', 'T:M2:R2:D4', 'T:M2:R2:D5', 'T:M2:R2:D7', 'T:M2:R2:D8')

surgery.annotate_observable(circuitry, 0, 'S:M1:R2:D1', 'S:M1:R2:D4', 'S:M1:R2:D7', 'M:M1:R2:D4', 'T:M2:R2:D1', 'T:M2:R2:D4', 'T:M2:R2:D7')

missing = circuitry.as_stim.missing_detectors()
print(f"Missing detectors : {len(missing)}")
for detector in missing:
    print(f"> Detector: {detector}")

display(Markdown(f"[Open in Crumble]({circuitry.as_stim.to_crumble_url()})"))

In [ ]:
for label, _ in qubits.measurements_index.items():
    print(f"Label {label} : {qubits.retrieve_measurement(label)}")

In [ ]:
circuitry.to_file("../generated/logical-teleportation")